# SpOC-3 Torso Decompositions — GPU run (QNE+GBFC hybrid) + CPU swarm

**Setup (one time):**
1. Create a Kaggle *Dataset* from `spoc3-gpu.zip` (Datasets → New Dataset → upload the zip; name it e.g. `spoc3-gpu`). Kaggle auto-extracts it.
2. New Notebook → **Add Input** → your `spoc3-gpu` dataset.
3. Settings → **Accelerator: GPU (T4 or P100)**. Internet ON (for pip, optional).
4. *Run All*. For a full-length unattended run use **Save Version → Save & Run All** (up to ~12 h GPU).

**What it does:** validates the GPU evaluator bit-for-bit against the official scorer, starts a small CPU GBFC++ swarm in the background, runs the `qnegbfc` hybrid (the winner's per-threshold neuroevolution + GBFC GBDT injection) on the GPU, then merges everything, re-verifies, and packs `results_spoc3.zip` for download.

**Thesis ablation:** set `USE_GBDT = False` below for the `--no-gbdt` control arm (same budget/seed).

In [ ]:
# ---- configuration ----------------------------------------------------
PROBLEM      = "small-graph"   # the instance we are pushing to #1
GPU_BUDGET_S = 36000           # ~10 h GPU search (leave margin in a 12 h session)
BATCH        = 2048            # GPU population per generation
SEED         = 42
USE_GBDT     = True            # False = --no-gbdt ablation arm
CPU_WORKERS  = 2               # background GBFC++ swarm workers (Kaggle has 4 vCPUs)
REQUIRE_GPU  = True            # hard-stop if the session has no GPU attached

In [ ]:
# ---- setup: locate the dataset, copy to a writable workspace ----------
import glob, os, shutil, subprocess, sys, zipfile

print("input tree (2 levels):")
for d in sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))[:20]:
    print(" ", d)

# 1) look for an extracted copy anywhere under /kaggle/input
hits = glob.glob("/kaggle/input/**/core.py", recursive=True)
src = os.path.dirname(hits[0]) if hits else None

# 2) fall back: the dataset kept the zip -- extract it ourselves
if src is None:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    assert zips, "neither core.py nor a .zip found under /kaggle/input -- attach the spoc3-gpu dataset as input"
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/kaggle/working/_extracted")
    hits = glob.glob("/kaggle/working/_extracted/**/core.py", recursive=True)
    assert hits, "zip extracted but core.py not inside"
    src = os.path.dirname(hits[0])

print("found repo at:", src)
REPO = "/kaggle/working/spoc3"
if not os.path.exists(os.path.join(REPO, "core.py")):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO)
print("repo:", REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "numba"], check=False)

# ---- GPU presence check (the #1 mistake is Accelerator: None) ----------
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv"], capture_output=True, text=True).stdout)
    HAS_GPU = True
else:
    HAS_GPU = False
    print("!" * 70)
    print("NO GPU IN THIS SESSION.")
    print("Notebook editor -> Settings (right sidebar) -> Accelerator -> GPU T4 x2 / P100,")
    print("then Save Version -> Save & Run All again.")
    print("!" * 70)
assert HAS_GPU or not REQUIRE_GPU, "GPU required but not attached -- enable the Accelerator and rerun (set REQUIRE_GPU=False to run CPU-only anyway)"

# build + self-test the C walk kernel (used by the CPU swarm)
print(subprocess.run([sys.executable, "tools/fastwalk.py", PROBLEM],
                     capture_output=True, text=True).stdout)

In [ ]:
# ---- gate: GPU evaluator must agree bit-for-bit with the official scorer
r = subprocess.run([sys.executable, "tools/validate_gpu.py", "--problem", PROBLEM,
                    "--batch", "256"], capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-1000:])
assert r.returncode == 0, "GPU validation failed — do not trust GPU scores"


In [ ]:
# ---- background CPU swarm (GBFC++ keeps polishing breakpoints) --------
swarm = subprocess.Popen(
    [sys.executable, "tools/gbfcpp_swarm.py", "--problem", PROBLEM,
     "--workers", str(CPU_WORKERS), "--waves", "999",
     "--rounds", "6", "--round-budget", "60", "--base-seed", "5000"],
    stdout=open("/kaggle/working/swarm.log", "w"), stderr=subprocess.STDOUT)
print("swarm pid:", swarm.pid, "(log: /kaggle/working/swarm.log)")

In [ ]:
# ---- MAIN: QNE + GBFC hybrid on the GPU (checkpoints itself) ----------
cmd = [sys.executable, "tools/qnegbfc.py", "--problem", PROBLEM,
       "--budget", str(GPU_BUDGET_S), "--batch", str(BATCH), "--seed", str(SEED)]
if not USE_GBDT:
    cmd.append("--no-gbdt")
print(" ".join(cmd), flush=True)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
print("\nGPU run done, rc =", p.returncode)

In [ ]:
# ---- merge, verify, package results ------------------------------------
swarm.terminate()
print(subprocess.run([sys.executable, "tools/portfolio.py", "--problems", PROBLEM],
                     capture_output=True, text=True).stdout[-2500:])
print(subprocess.run([sys.executable, "tools/verify_submission.py",
                      f"submissions/{PROBLEM}/portfolio.json"],
                     capture_output=True, text=True).stdout[-1200:])

shutil.make_archive("/kaggle/working/results_spoc3", "zip",
                    root_dir=REPO, base_dir="submissions")
print("\nDownload from the notebook Output tab: results_spoc3.zip")
print("tail of swarm log:")
print(open("/kaggle/working/swarm.log").read()[-1500:])

## Bringing the results home

1. Download `results_spoc3.zip` from the notebook's **Output** tab.
2. On your Mac, extract it and copy the JSONs **into** the repo (they only add, never overwrite your better fronts after the merge):
   ```bash
   cd "Torso Decompositions"
   unzip -o ~/Downloads/results_spoc3.zip -d /tmp/kaggle_results
   cp /tmp/kaggle_results/submissions/small-graph/qnegbfc*.json submissions/small-graph/
   cp /tmp/kaggle_results/submissions/small-graph/gbfcpp*.json  submissions/small-graph/ 2>/dev/null
   python3 tools/portfolio.py --problems small-graph
   python3 tools/verify_submission.py submissions/small-graph/portfolio.json
   ```
3. If the verified score is below −1,829,919: **leaderboard beaten** — re-check the live ESA board before claiming it in the thesis.

**Notes**
- Session dies? Everything is checkpointed; the zip in Output keeps the last saved state. Re-run the notebook — it resumes from the dataset's banked fronts (to resume from *newer* checkpoints, update the dataset with your latest local `submissions/`).
- For the thesis ablation, run a second version with `USE_GBDT = False` and compare `qnegbfc` vs `qne_wide` stems at equal budget.
- `medium-graph` / `large-graph`: change `PROBLEM`; the same pipeline applies (gaps there are larger — see THESIS §11).